# FEATURE ENGINEERING V1

Pipeline:

`nba_full_dataset.csv` → Fix issues → EMA Rolling → Ghép 2 đội → Difference Features → Train/Test Split → **Model-ready dataset**

# 0. Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r'C:\Code_AI\CS114-FinalTerm\code\nba_data\nba_full_dataset.csv')
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

print(f'Dataset gốc: {df.shape[0]} dòng × {df.shape[1]} cột')
print(f'Số trận: {df["GAME_ID"].nunique()}')
print(f'Mùa giải: {sorted(df["SEASON"].unique())}')
print(f'\nCác cột hiện có:')
print(list(df.columns))

Dataset gốc: 12300 dòng × 36 cột
Số trận: 6150
Mùa giải: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

Các cột hiện có:
['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'SEASON', 'IS_HOME', 'WIN', 'REST_DAYS', 'IS_B2B', 'GAMES_PLAYED_SEASON', 'CURRENT_WIN_PCT', 'WIN_STREAK']


# 1. Xử lý vấn đề phát hiện từ EDA

- `FT_PCT` có 1 missing value -> fill 0
- Trận neutral site -> Cả 2 đội đều IS_HOME = 0 -> loại

In [2]:
# 1.1 - Fill missing
df['FT_PCT'] = df['FT_PCT'].fillna(0)

# 1.2 - Cap REST_DAYS
print(f'REST_DAYS trước khi cap: min={df["REST_DAYS"].min()}, max={df["REST_DAYS"].max()}')
df['REST_DAYS'] = df['REST_DAYS'].clip(upper=10)
print(f'REST_DAYS sau khi cap:   min={df["REST_DAYS"].min()}, max={df["REST_DAYS"].max()}')

# 1.3 — Loại trận neutral site (cả 2 đội IS_HOME=0)
game_home_count = df.groupby('GAME_ID')['IS_HOME'].sum()
neutral_games = game_home_count[game_home_count != 1].index
n_neutral = len(neutral_games)
df = df[~df['GAME_ID'].isin(neutral_games)]

print(f'\nLoại {n_neutral} trận neutral site (cả 2 đội IS_HOME=0)')
print(f'Dataset sau xử lý: {df.shape[0]} dòng, {df["GAME_ID"].nunique()} trận')

# Xác nhận: mỗi trận có đúng 1 HOME + 1 AWAY
home_count = (df['IS_HOME'] == 1).sum()
away_count = (df['IS_HOME'] == 0).sum()
print(f'HOME: {home_count}, AWAY: {away_count}')

REST_DAYS trước khi cap: min=1.0, max=9.0
REST_DAYS sau khi cap:   min=1.0, max=9.0

Loại 10 trận neutral site (cả 2 đội IS_HOME=0)
Dataset sau xử lý: 12280 dòng, 6140 trận
HOME: 6140, AWAY: 6140


In [ ]:
# 1.3 — Loại trận neutral site (cả 2 đội IS_HOME=0)
game_home_count = df.groupby('GAME_ID')['IS_HOME'].sum()
neutral_games = game_home_count[game_home_count != 1].index
if len(neutral_games) > 0:
    print(f"\n--- PHÁT HIỆN {len(neutral_games)} TRẬN NEUTRAL SITE ---")
    neutral_df = df[df['GAME_ID'].isin(neutral_games)]
    cols_to_view = ['GAME_DATE', 'GAME_ID', 'TEAM_ID', 'MATCHUP', 'IS_HOME']
    print(neutral_df[cols_to_view].sort_values(['GAME_DATE', 'GAME_ID']).to_string(index=False))
else:
    print("\n--- KHÔNG CÓ TRẬN NEUTRAL SITE NÀO ---")


--- PHÁT HIỆN 10 TRẬN NEUTRAL SITE ---
 GAME_DATE  GAME_ID    TEAM_ID   MATCHUP  IS_HOME
2024-11-02 22400147 1610612748 MIA @ WAS        0
2024-11-02 22400147 1610612764 MIA @ WAS        0
2024-12-14 22401229 1610612737 ATL @ MIL        0
2024-12-14 22401229 1610612749 ATL @ MIL        0
2024-12-14 22401230 1610612745 HOU @ OKC        0
2024-12-14 22401230 1610612760 HOU @ OKC        0
2025-01-23 22400621 1610612754 SAS @ IND        0
2025-01-23 22400621 1610612759 SAS @ IND        0
2025-01-25 22400633 1610612754 IND @ SAS        0
2025-01-25 22400633 1610612759 IND @ SAS        0
2025-11-01 22500147 1610612742 DAL @ DET        0
2025-11-01 22500147 1610612765 DAL @ DET        0
2025-12-13 22501229 1610612752 NYK @ ORL        0
2025-12-13 22501229 1610612753 NYK @ ORL        0
2025-12-13 22501230 1610612759 SAS @ OKC        0
2025-12-13 22501230 1610612760 SAS @ OKC        0
2026-01-15 22500578 1610612753 MEM @ ORL        0
2026-01-15 22500578 1610612763 MEM @ ORL        0
2026-01-18

## 2. EMA Rolling Averages (Exponential Moving Average)

In [3]:
EMA_FEATURES = ['PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT',
                'OREB', 'DREB', 'AST', 'STL', 'BLK', 'TOV']
EMA_SPAN = 5

print(f'Features tính EMA: {EMA_FEATURES}')
print(f'EMA span: {EMA_SPAN} trận')

Features tính EMA: ['PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'OREB', 'DREB', 'AST', 'STL', 'BLK', 'TOV']
EMA span: 5 trận


In [4]:
# Sắp xếp theo đội + mùa + thời gian
df = df.sort_values(['TEAM_ID', 'SEASON', 'GAME_DATE']).reset_index(drop=True)

# Tính EMA cho từng feature
for feat in EMA_FEATURES:
    col_name = f'EMA_{feat}'
    # shift(1): chỉ dùng dữ liệu TRƯỚC trận hiện tại (no leakage)
    shifted = df.groupby(['TEAM_ID', 'SEASON'])[feat].shift(1)
    # EMA: trọng số giảm dần theo hàm mũ, trận gần có trọng số cao hơn
    df[col_name] = shifted.groupby([df['TEAM_ID'], df['SEASON']]).transform(
        lambda x: x.ewm(span=EMA_SPAN, adjust=False).mean()
    )

print(f'Số cột mới: {len(EMA_FEATURES)} (EMA_PTS, EMA_FG_PCT, ...)')

Số cột mới: 10 (EMA_PTS, EMA_FG_PCT, ...)


In [5]:
# Kiểm tra: 8 trận đầu của 1 đội
sample_team = df['TEAM_ABBREVIATION'].iloc[0]
sample_season = df['SEASON'].iloc[0]
sample_mask = (df['TEAM_ABBREVIATION'] == sample_team) & (df['SEASON'] == sample_season)

print(f'Ví dụ: 8 trận đầu của {sample_team} mùa {sample_season}')
print('-' * 90)
check_cols = ['GAME_DATE', 'PTS', 'EMA_PTS', 'FG_PCT', 'EMA_FG_PCT', 'AST', 'EMA_AST']
print(df[sample_mask].head(8)[check_cols].to_string(index=False))

print(f'\n→ Trận 1: NaN (chưa có lịch sử — sẽ bị loại khi ghép đội)')
print(f'→ Trận 2: EMA = giá trị trận 1 (chỉ có 1 điểm dữ liệu)')
print(f'→ Trận 3+: EMA bắt đầu smooth — trận gần ảnh hưởng nhiều hơn')

Ví dụ: 8 trận đầu của ATL mùa 2021-22
------------------------------------------------------------------------------------------
 GAME_DATE  PTS    EMA_PTS  FG_PCT  EMA_FG_PCT  AST   EMA_AST
2021-10-21  113        NaN   0.479         NaN   31       NaN
2021-10-23   95 113.000000   0.384    0.479000   20 31.000000
2021-10-25  122 107.000000   0.511    0.447333   24 27.333333
2021-10-27  102 112.000000   0.417    0.468556   21 26.222222
2021-10-28  111 108.666667   0.545    0.451370   26 24.481481
2021-10-30   94 109.444444   0.379    0.482580   24 24.987654
2021-11-01  118 104.296296   0.458    0.448053   24 24.658436
2021-11-03  108 108.864198   0.436    0.451369   23 24.438957

→ Trận 1: NaN (chưa có lịch sử — sẽ bị loại khi ghép đội)
→ Trận 2: EMA = giá trị trận 1 (chỉ có 1 điểm dữ liệu)
→ Trận 3+: EMA bắt đầu smooth — trận gần ảnh hưởng nhiều hơn


# 3. Ghép 2 đội thành 1 dòng

In [6]:
# Định nghĩa các features của mỗi đội sẽ đưa vào model
team_features = (
    [f'EMA_{f}' for f in EMA_FEATURES]  # 10 EMA features
    + ['CURRENT_WIN_PCT', 'WIN_STREAK', 'REST_DAYS', 'IS_B2B']  # 4 contextual
)

print(f'Features mỗi đội: {len(team_features)} cột')
for f in team_features:
    print(f'  - {f}')

Features mỗi đội: 14 cột
  - EMA_PTS
  - EMA_FG_PCT
  - EMA_FG3_PCT
  - EMA_FT_PCT
  - EMA_OREB
  - EMA_DREB
  - EMA_AST
  - EMA_STL
  - EMA_BLK
  - EMA_TOV
  - CURRENT_WIN_PCT
  - WIN_STREAK
  - REST_DAYS
  - IS_B2B


In [7]:
# Tách HOME và AWAY
home_df = df[df['IS_HOME'] == 1].copy()
away_df = df[df['IS_HOME'] == 0].copy()

# Các cột chung (metadata)
game_cols = ['GAME_ID', 'GAME_DATE', 'SEASON']

# Rename: thêm prefix HOME_ / AWAY_
home_rename = {f: f'HOME_{f}' for f in team_features}
home_rename['TEAM_ABBREVIATION'] = 'HOME_TEAM'
home_rename['WIN'] = 'HOME_WIN'

away_rename = {f: f'AWAY_{f}' for f in team_features}
away_rename['TEAM_ABBREVIATION'] = 'AWAY_TEAM'

home_df = home_df.rename(columns=home_rename)
away_df = away_df.rename(columns=away_rename)

# Chọn cột cần giữ
home_cols = game_cols + ['HOME_TEAM', 'HOME_WIN'] + [f'HOME_{f}' for f in team_features]
away_cols = ['GAME_ID', 'AWAY_TEAM'] + [f'AWAY_{f}' for f in team_features]

# Merge theo GAME_ID
merged = home_df[home_cols].merge(away_df[away_cols], on='GAME_ID', how='inner')

# Loại NaN (trận đầu mùa không có EMA)
before = len(merged)
merged = merged.dropna().reset_index(drop=True)
print(f'Sau ghép: {before} trận → loại {before - len(merged)} NaN → còn {len(merged)} trận')
print(f'Shape: {merged.shape}')

Sau ghép: 6140 trận → loại 79 NaN → còn 6061 trận
Shape: (6061, 34)


In [8]:
# Kiểm tra
print('Ví dụ 3 trận đầu:')
print('=' * 90)
for idx, row in merged.head(3).iterrows():
    print(f'\n{row["HOME_TEAM"]} (nhà) vs {row["AWAY_TEAM"]} (khách) | {row["GAME_DATE"]}')
    print(f'  HOME: EMA_PTS={row["HOME_EMA_PTS"]:.1f}, WIN_PCT={row["HOME_CURRENT_WIN_PCT"]:.3f}')
    print(f'  AWAY: EMA_PTS={row["AWAY_EMA_PTS"]:.1f}, WIN_PCT={row["AWAY_CURRENT_WIN_PCT"]:.3f}')
    print(f'  → {"HOME thắng" if row["HOME_WIN"]==1 else "AWAY thắng"}')

print(f'\nLabel: HOME thắng {merged["HOME_WIN"].mean():.1%}, AWAY thắng {(1-merged["HOME_WIN"]).mean():.1%}')

Ví dụ 3 trận đầu:

ATL (nhà) vs DET (khách) | 2021-10-25 00:00:00
  HOME: EMA_PTS=107.0, WIN_PCT=0.500
  AWAY: EMA_PTS=86.0, WIN_PCT=0.000
  → HOME thắng

ATL (nhà) vs WAS (khách) | 2021-11-01 00:00:00
  HOME: EMA_PTS=104.3, WIN_PCT=0.500
  AWAY: EMA_PTS=113.3, WIN_PCT=0.833
  → HOME thắng

ATL (nhà) vs UTA (khách) | 2021-11-04 00:00:00
  HOME: EMA_PTS=108.6, WIN_PCT=0.500
  AWAY: EMA_PTS=112.4, WIN_PCT=0.857
  → AWAY thắng

Label: HOME thắng 55.4%, AWAY thắng 44.6%


# 4. Difference Features (HOME - AWAY)

In [9]:
DIFF_FEATURES = {
    # 10 EMA features
    'EMA_PTS': 'DIFF_PTS',
    'EMA_FG_PCT': 'DIFF_FG_PCT',
    'EMA_FG3_PCT': 'DIFF_FG3_PCT',
    'EMA_FT_PCT': 'DIFF_FT_PCT',
    'EMA_OREB': 'DIFF_OREB',
    'EMA_DREB': 'DIFF_DREB',
    'EMA_AST': 'DIFF_AST',
    'EMA_STL': 'DIFF_STL',
    'EMA_BLK': 'DIFF_BLK',
    'EMA_TOV': 'DIFF_TOV',
    # 3 contextual features
    'CURRENT_WIN_PCT': 'DIFF_WIN_PCT',
    'WIN_STREAK': 'DIFF_WIN_STREAK',
    'REST_DAYS': 'DIFF_REST_DAYS',
}

In [10]:
for feat, diff_name in DIFF_FEATURES.items():
    merged[diff_name] = merged[f'HOME_{feat}'] - merged[f'AWAY_{feat}']

print(f'Đã tạo {len(DIFF_FEATURES)} difference features:')
print('-' * 60)
for diff_name in DIFF_FEATURES.values():
    val = merged[diff_name]
    print(f'  {diff_name:20s}: mean={val.mean():+.4f}, std={val.std():.4f}')

Đã tạo 13 difference features:
------------------------------------------------------------
  DIFF_PTS            : mean=-0.2177, std=9.4608
  DIFF_FG_PCT         : mean=-0.0003, std=0.0406
  DIFF_FG3_PCT        : mean=-0.0014, std=0.0565
  DIFF_FT_PCT         : mean=-0.0005, std=0.0734
  DIFF_OREB           : mean=+0.0105, std=3.1327
  DIFF_DREB           : mean=+0.0211, std=4.0666
  DIFF_AST            : mean=-0.0621, std=4.1317
  DIFF_STL            : mean=-0.0191, std=2.2086
  DIFF_BLK            : mean=-0.0131, std=1.8879
  DIFF_TOV            : mean=+0.0038, std=2.8213
  DIFF_WIN_PCT        : mean=-0.0026, std=0.2609
  DIFF_WIN_STREAK     : mean=+0.0541, std=4.7459
  DIFF_REST_DAYS      : mean=+0.0813, std=0.9999


# 5. Định nghĩa bộ Features cho Model

In [11]:
# Bộ features cho model
feature_set = list(DIFF_FEATURES.values()) + ['HOME_IS_B2B', 'AWAY_IS_B2B']

LABEL = 'HOME_WIN'

print(f'Bộ features: {len(feature_set)} cột')
print(f'Label: {LABEL}')
print(f'\nDanh sách features:')
for i, f in enumerate(feature_set, 1):
    print(f'  {i:2d}. {f}')

Bộ features: 15 cột
Label: HOME_WIN

Danh sách features:
   1. DIFF_PTS
   2. DIFF_FG_PCT
   3. DIFF_FG3_PCT
   4. DIFF_FT_PCT
   5. DIFF_OREB
   6. DIFF_DREB
   7. DIFF_AST
   8. DIFF_STL
   9. DIFF_BLK
  10. DIFF_TOV
  11. DIFF_WIN_PCT
  12. DIFF_WIN_STREAK
  13. DIFF_REST_DAYS
  14. HOME_IS_B2B
  15. AWAY_IS_B2B


# 6. Train / Test split (2 lần chia)

- Lần 1 (Đánh giá): Train 4 mùa (2021 - 2025) | Test (2025 - 2026) -> đo accuracy, F1
- Lần 2 (deploy): Train tất cả -> model mạnh nhất dự đoán cho trận mới

In [ ]:
eval_train_seasons = ['2021-22', '2022-23', '2023-24', '2024-25']
eval_test_season = '2025-26'

eval_train = merged[merged['SEASON'].isin(eval_train_seasons)].copy()
eval_test = merged[merged['SEASON'] == eval_test_season].copy()

print('LẦN 1 — ĐÁNH GIÁ MODEL (BÁO CÁO)')
print('=' * 50)
print(f'Train: {len(eval_train)} trận | {eval_train["GAME_DATE"].min().date()} → {eval_train["GAME_DATE"].max().date()}')
print(f'Test:  {len(eval_test)} trận | {eval_test["GAME_DATE"].min().date()} → {eval_test["GAME_DATE"].max().date()}')


print(f'Train HOME_WIN: {eval_train["HOME_WIN"].mean():.1%} | Test HOME_WIN: {eval_test["HOME_WIN"].mean():.1%}')

LẦN 1 — ĐÁNH GIÁ MODEL (BÁO CÁO)
Train: 4852 trận | 2021-10-22 → 2025-04-13
Test:  1209 trận | 2025-10-24 → 2026-04-12
Train HOME_WIN: 55.4% | Test HOME_WIN: 55.3%


In [ ]:
final_train = merged.copy()

print('\nLẦN 2 — MODEL CUỐI CÙNG (DEMO)')
print('=' * 50)
print(f'Train: {len(final_train)} trận (TẤT CẢ dữ liệu)')
print(f'Train: {final_train["GAME_DATE"].min().date()} → {final_train["GAME_DATE"].max().date()}')
print(f'→ Predict: các trận SAU {final_train["GAME_DATE"].max().date()}')


LẦN 2 — MODEL CUỐI CÙNG (DEMO)
Train: 6061 trận (TẤT CẢ dữ liệu)
Train: 2021-10-22 → 2026-04-12
→ Predict: các trận SAU 2026-04-12


In [14]:
# Tách X, y
# Lần 1
X_eval_train = eval_train[feature_set]
X_eval_test  = eval_test[feature_set]
y_eval_train = eval_train[LABEL]
y_eval_test  = eval_test[LABEL]

# Lần 2
X_final = final_train[feature_set]
y_final = final_train[LABEL]

print('LẦN 1:')
print(f'  train: X={X_eval_train.shape}, y={y_eval_train.shape}')
print(f'  test:  X={X_eval_test.shape}, y={y_eval_test.shape}')
print(f'\nLẦN 2:')
print(f'  train: X={X_final.shape}, y={y_final.shape}')

LẦN 1:
  train: X=(4852, 15), y=(4852,)
  test:  X=(1209, 15), y=(1209,)

LẦN 2:
  train: X=(6061, 15), y=(6061,)


# 7. Lưu dataset

- `eval_train_v1.csv` (Tập huấn luyện 4 mùa giải): Dùng file này làm đầu vào (`X_test`, `y_test`) để train cho thuật toán 
- `eval_test_v1.csv` (Tập test mùa 2025-2026): Sau khi mô hình học xong từ file train, bạn dùng file này (X_test, y_test) để bắt nó làm bài kiểm tra. Vì file test chứa dữ liệu hoàn toàn xa lạ với mô hình, kết quả sinh ra (Accuracy, F1-Score, Confusion Matrix) sẽ hoàn toàn trung thực. Đây chính là những con số và biểu đồ bạn sẽ chụp lại và dán vào báo cáo đồ án nộp cho giảng viên.
- `nba_model_ready_v1.csv` (Toàn bộ dữ liệu 5 mùa):File này chứa tất cả dữ liệu sạch sẽ nhất từ năm 2021 đến tận ngày hôm qua. Sau khi báo cáo xong các số liệu ở Nhóm 1, bạn sẽ gom hết dữ liệu vào file này để train lại mô hình một lần cuối cùng. Việc học toàn bộ dữ liệu giúp mô hình nắm bắt được phong độ mới nhất của các đội. File model sinh ra từ đây sẽ được lưu lại (file .pkl hoặc .joblib) để nhúng vào trang web Streamlit, dùng để dự đoán trực tiếp các trận đấu ngày mai.
- `feature_config_v1.json`: lưu trữ toàn bộ các thông số như EMA_SPAN = 5, danh sách 15 cột features, và cột label. Nó có 2 tác dụng:
    - Quản lý phiên bản: Vài ngày nữa khi bạn làm bản V2 (thêm Elo) hay V3 (Four Factors), việc có các file JSON này giúp bạn so sánh chính xác cấu hình nào mang lại kết quả tốt nhất mà không cần mở lại code đọc.
    - Hỗ trợ Web Demo: Khi Web Demo cào dữ liệu lịch thi đấu ngày mai về, nó phải biết cần tính toán đặc trưng gì và ráp các cột theo thứ tự nào để đưa vào mô hình. Script của Web chỉ cần đọc file JSON này là biết chính xác cấu trúc dữ liệu mô hình đang cần.

In [ ]:
import os

save_dir = 'output'
os.makedirs(save_dir, exist_ok=True)

# Lưu các file CSV
merged.to_csv(f'{save_dir}/nba_model_ready_v1.csv', index=False)
eval_train.to_csv(f'{save_dir}/eval_train_v1.csv', index=False)
eval_test.to_csv(f'{save_dir}/eval_test_v1.csv', index=False)

print(f'Đã lưu 3 file CSV vào thư mục: {save_dir}/')

# Lưu file cấu hình JSON
feature_config = {
    'version': 'V1',
    'method': 'EMA',
    'ema_span': EMA_SPAN,
    'ema_features': EMA_FEATURES,
    'diff_features': DIFF_FEATURES,
    'team_features': team_features,
    'feature_set': feature_set,
    'label': LABEL,
    'eval_train_seasons': eval_train_seasons,
    'eval_test_season': eval_test_season,
}

with open(f'{save_dir}/feature_config_v1.json', 'w', encoding='utf-8') as f:
    json.dump(feature_config, f, indent=2, ensure_ascii=False)
    
print(f'Đã lưu file config vào: {save_dir}/feature_config_v1.json')

Đã lưu 3 file CSV vào thư mục: output/
Đã lưu file config vào: output/feature_config_v1.json
